# 01_data_preprocessing.ipynb

이 노트북은 NYC Yellow Taxi 데이터를 Polars를 사용하여 전처리하고, 3분 단위의 택시 수요 데이터를 생성합니다.

## 1. 필요한 라이브러리 임포트

In [2]:
import polars as pl
import os
from datetime import datetime


## 2. 데이터 로드 (Lazy Loading)

`data/raw` 폴더에 있는 모든 월별 Parquet 파일을 Lazy Loading으로 읽습니다.
실제 데이터셋은 여기 없지만, 아래 코드는 파일들이 `data/raw`에 있다고 가정하고 실행됩니다.

In [3]:
# data/raw 폴더가 존재하지 않거나 비어있을 경우를 대비한 가이드라인
# 실제 데이터를 다운로드하여 data/raw 폴더에 넣어주세요.
# 예시: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

raw_data_path = "../data/raw/*.parquet"

# 파일이 존재하는지 확인 (실제 실행 시 필요)
# import glob
# if not glob.glob(raw_data_path):
#     print("Error: No parquet files found in data/raw. Please download the data first.")
# else:
#     print("Parquet files found. Proceeding with lazy loading.")

lf = pl.scan_parquet(raw_data_path)
print("LazyFrame created. Schema preview (first 5 columns):")
print(lf.collect_schema())


LazyFrame created. Schema preview (first 5 columns):
Schema({'DOLocationID': Int64, 'PULocationID': Int64, 'RatecodeID': Int64, 'VendorID': Int64, 'congestion_surcharge': Float64, 'extra': Float64, 'fare_amount': Float64, 'improvement_surcharge': Float64, 'mta_tax': Float64, 'passenger_count': Int64, 'payment_type': Int64, 'store_and_fwd_flag': String, 'tip_amount': Float64, 'tolls_amount': Float64, 'total_amount': Float64, 'tpep_dropoff_datetime': Datetime(time_unit='us', time_zone=None), 'tpep_pickup_datetime': Datetime(time_unit='us', time_zone=None), 'trip_distance': Float64, 'Airport_fee': Float64, 'trip_id': Int64})


## 3. 데이터 전처리

다음 단계를 수행합니다:
1.  **컬럼 정규화**: `Airport_fee` 컬럼이 있을 경우 `airport_fee`로 이름을 통일하고, 결측치는 0.0으로 채웁니다.
2.  **날짜 및 조건 필터링**: `tpep_pickup_datetime`을 기준으로 2023-01-01부터 2025-11-30까지의 데이터를 선택하고, `passenger_count > 0` 및 `trip_distance > 0` 조건을 만족하는 행만 필터링합니다.
3.  **수요 집계**: `tpep_pickup_datetime`을 3분 단위로 자르고 (`3m`), `pickup_time`과 `PULocationID`를 기준으로 그룹화하여 각 그룹의 행 수를 `demand`로 집계합니다.

In [4]:
processed_lf = lf.with_columns(
    # Airport_fee 컬럼을 airport_fee로 통일하고 결측치는 0.0으로 채움
    pl.col("Airport_fee").fill_null(0.0).alias("airport_fee")
).select(
    # 필요한 컬럼만 선택하고, pickup_datetime을 날짜 범위 필터링에 사용할 수 있도록 캐스팅
    pl.col("tpep_pickup_datetime").cast(pl.Datetime).alias("pickup_datetime"),
    "PULocationID",
    "passenger_count",
    "trip_distance",
    "airport_fee" # 정규화된 airport_fee 사용
).filter(
    (pl.col("pickup_datetime") >= datetime(2023, 1, 1))
    & (pl.col("pickup_datetime") <= datetime(2025, 11, 30))
    & (pl.col("passenger_count") > 0)
    & (pl.col("trip_distance") > 0)
).with_columns(
    # 3분 단위로 시간 자르기
    pl.col("pickup_datetime").dt.truncate("3m").alias("pickup_time")
).group_by(["pickup_time", "PULocationID"]).agg(
    # 각 3분 단위, 위치별 수요 집계
    pl.len().alias("demand")
).sort(["pickup_time", "PULocationID"])

print("Processed LazyFrame created. The result will be written to parquet in the next step.")


Processed LazyFrame created. The result will be written to parquet in the next step.


## 4. 결과 저장

전처리 및 집계된 데이터를 `data/processed/aggregated_demand_3min.parquet` 파일로 저장합니다.

In [5]:
# LazyFrame을 실행하여 Eager DataFrame으로 변환
print("Collecting aggregated data...")
aggregated_df = processed_lf.collect()

# Upsampling을 통해 비어있는 시간대를 0으로 채워 완전한 시계열 데이터 생성
print("Upsampling to fill missing time buckets...")
complete_df = (
    aggregated_df
    .upsample(time_column="pickup_time", every="3m", group_by="PULocationID")
    .with_columns([
        pl.col("PULocationID").forward_fill(),
        pl.col("demand").fill_null(0).cast(pl.UInt32) # 수요가 없는 곳은 0으로 채움
    ])
)

# 최종 결과물 저장
output_path = "../data/processed/aggregated_demand_3min.parquet"
print(f"Saving complete time series data to {output_path}...")
complete_df.write_parquet(output_path)

print("Data processing complete and saved.")
print(f"Data saved to {output_path}")

Upsampling to fill missing time buckets...
Saving complete time series data to ../data/processed/aggregated_demand_3min.parquet...
Data processing complete and saved.
Data saved to ../data/processed/aggregated_demand_3min.parquet


## 5. 저장된 데이터 확인 (선택 사항)

In [6]:
try:
    check_df = pl.read_parquet(output_path)
    print("Successfully re-read saved parquet file. Head:")
    print(check_df.head())
except Exception as e:
    print(f"Error reading saved file: {e}")


Successfully re-read saved parquet file. Head:
shape: (5, 3)
┌─────────────────────┬──────────────┬────────┐
│ pickup_time         ┆ PULocationID ┆ demand │
│ ---                 ┆ ---          ┆ ---    │
│ datetime[μs]        ┆ i64          ┆ u32    │
╞═════════════════════╪══════════════╪════════╡
│ 2023-01-01 00:00:00 ┆ 170          ┆ 2      │
│ 2023-01-01 00:03:00 ┆ 170          ┆ 3      │
│ 2023-01-01 00:06:00 ┆ 170          ┆ 7      │
│ 2023-01-01 00:09:00 ┆ 170          ┆ 5      │
│ 2023-01-01 00:12:00 ┆ 170          ┆ 5      │
└─────────────────────┴──────────────┴────────┘
